# Aula 08: Geracao e Deteccao de Trojans de Hardware

PCS5917 IA Adversarial, EP-USP — Grupo C

Geramos, com um LLM, **10 refinamentos legitimos** e **10 trojans de hardware** sobre um
*golden model* aberto de AES (`secworks/aes`, Verilog), no estilo dos benchmarks publicos
**Trust-Hub AES-T**. Compilamos e sintetizamos as amostras (desafio opcional) com `iverilog`
e `yosys`, montamos a planilha rotulada e a versao anonimizada, e usamos um **segundo LLM de
familia diferente** como detector. O exercicio e um estudo de **purple team**: a mesma base
serve para red team (gerar) e blue team (detectar).

- **Gerador:** `Qwen2.5-Coder-14B-Instruct-AWQ` (forte em Verilog).
- **Detector:** familia diferente do gerador (DeepSeek-Coder) — requisito do grupo: quem gera nao detecta.
- `RUN=False` (padrao) carrega os artefatos ja salvos em `outros/aula-08/`; `RUN=True` reexecuta na GPU (T4 do Colab).


## 1. Setup (repo, deps, golden model)

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
RUN = False  # True = (re)gera na GPU T4; False = usa os artefatos salvos em outros/aula-08
from pathlib import Path

if IN_COLAB and not Path("/content/portfolio-pcs5917").exists():
    os.system("git clone --depth 1 https://github.com/LuizZucchi/portfolio-pcs5917 /content/portfolio-pcs5917")
ROOT = Path("/content/portfolio-pcs5917") if IN_COLAB else Path("..").resolve()
A08 = ROOT / "outros" / "aula-08"
sys.path.insert(0, str(A08))
OUT = A08; OUT.mkdir(parents=True, exist_ok=True)
GOLDEN = ROOT / "outros" / "aula-08" / "_golden"  # clone do secworks/aes

import taxonomy as TX
import prompts as PR
print("RUN =", RUN, "| COLAB =", IN_COLAB, "| ROOT =", ROOT)


In [ ]:
# Instala dependencias e clona o golden model apenas quando RUN=True.
if RUN:
    os.system("pip -q install 'vllm==0.6.3.post1' >/dev/null")
    os.system("apt-get -qq install -y iverilog yosys >/dev/null")
    if not GOLDEN.exists():
        os.system(f"git clone --depth 1 https://github.com/secworks/aes {GOLDEN}")
    print("deps ok")


O golden model e o `secworks/aes`: nucleo AES-128/256 em Verilog com testbench. As
modificacoes recaem sobre `aes_core.v` (e modulos vizinhos). O testbench `src/tb/tb_aes.v`
valida contra os vetores de teste do NIST — um trojan *furtivo* deve passar nele enquanto
dormente.

In [ ]:
# Localiza o modulo-alvo e o testbench do golden model.
RTL = GOLDEN / "src" / "rtl"
TB  = GOLDEN / "src" / "tb" / "tb_aes.v"
def read_module(name):
    for f in sorted(RTL.glob("*.v")):
        t = f.read_text()
        import re
        if re.search(rf"\bmodule\s+{name}\b", t):
            return t
    raise FileNotFoundError(name)
BASE = read_module(TX.TARGET_MODULE) if RUN else ""
if RUN: print(TX.TARGET_MODULE, "=", len(BASE), "chars")


## 2. Modelos (gerador e detector)

In [ ]:
# Carregamento preguicoso do vLLM. Gerador e detector rodam em FASES separadas
# (um de cada vez) para caber na T4 (16 GB): gera tudo, libera, carrega o detector.
GEN_MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"
DET_MODEL = "casperhansen/deepseek-coder-v2-lite-instruct-awq"  # familia diferente
import gc

def make_llm(model_id):
    from vllm import LLM
    return LLM(model=model_id, quantization="awq", dtype="float16",
               gpu_memory_utilization=0.92, max_model_len=8192,
               enforce_eager=True, trust_remote_code=True)

def free(llm):
    import torch, contextlib
    with contextlib.suppress(Exception):
        del llm
    gc.collect()
    with contextlib.suppress(Exception):
        torch.cuda.empty_cache()

def chat(llm, system, user, max_tokens=4096, temperature=0.4):
    from vllm import SamplingParams
    tok = llm.get_tokenizer()
    msgs = [{"role":"system","content":system},{"role":"user","content":user}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    sp = SamplingParams(temperature=temperature, top_p=0.9, max_tokens=max_tokens)
    return llm.generate([prompt], sp)[0].outputs[0].text

import re
def extract_verilog(text):
    m = re.search(r"```(?:verilog|systemverilog)?\s*(.*?)```", text, re.S)
    code = m.group(1) if m else text
    return code.strip()

REFUSAL = re.compile(r"\b(I can('|no)?t|cannot|won'?t|not able|against .*polic|"
                     r"I'm sorry|as an ai|refus)\b", re.I)
def looks_refusal(text):
    head = text.strip()[:400]
    return bool(REFUSAL.search(head)) and "module" not in text


## 3. Geracao legitima (10 amostras)

In [ ]:
import json
def gen_legit(llm):
    rows = []
    for spec in TX.LEGIT:
        _id, kind, desc = spec
        out = chat(llm, PR.SYS_GEN, PR.prompt_legit(spec, BASE), temperature=0.3)
        rows.append({"id": _id, "label": "legit", "kind": kind, "desc": desc,
                     "step": None, "code": extract_verilog(out)})
        print(_id, kind, "->", len(rows[-1]["code"]), "chars")
    return rows


## 4. Geracao de trojans (escada de abordagens)

Para cada trojan subimos a *escada* `direct -> academic -> decompose -> pair` ate o gerador
produzir codigo plausivel (sem recusa e com um `module`). Registramos em `step` qual degrau
funcionou — e o analogo de hardware do que fizemos na Aula 07 com jailbreaks de prompt.

In [ ]:
STEP_NAME = {0:"direct",1:"academic",2:"decompose",3:"pair"}
def gen_trojan(llm):
    rows = []
    for spec in TX.TROJAN:
        _id, trig, pay, desc = spec
        chosen, code_out, step_used = None, "", None
        for step in range(4):
            out = chat(llm, PR.SYS_GEN, PR.prompt_trojan(spec, BASE, step=step),
                       temperature=0.3 + 0.15*step)
            code_out = extract_verilog(out)
            if not looks_refusal(out) and "module" in code_out:
                step_used = step; break
        rows.append({"id": _id, "label": "trojan", "trigger": trig, "payload": pay,
                     "desc": desc, "step": step_used, "step_name": STEP_NAME.get(step_used),
                     "code": code_out})
        print(_id, trig, pay, "-> step", step_used, STEP_NAME.get(step_used))
    return rows


In [ ]:
# Fase de GERACAO: carrega o gerador, gera as 20 amostras, salva e libera a GPU.
if RUN:
    gen = make_llm(GEN_MODEL)
    legit = gen_legit(gen)
    trj   = gen_trojan(gen)
    free(gen)
    samples = legit + trj
    (OUT/"samples_raw.json").write_text(json.dumps(samples, indent=1, ensure_ascii=False))
    print("salvo samples_raw.json:", len(samples), "amostras")
else:
    samples = json.loads((OUT/"samples_raw.json").read_text())
    print("carregado samples_raw.json:", len(samples), "amostras")


## 5. Verificacao: compila + testbench + sintese (desafio opcional)

In [ ]:
# iverilog + testbench do golden model; yosys para contagem de celulas (overhead).
if RUN:
    from verify import verify_sample
    for s in samples:
        name = TX.TARGET_MODULE
        v = verify_sample(RTL, TB, name, s["code"])
        s.update({"compiles": v["compiles"], "tb_pass": v["tb_pass"],
                  "synth_ok": v["synth_ok"], "cells": v["cells"]})
        print(s["id"], "compila", v["compiles"], "| tb_pass", v["tb_pass"], "| cells", v["cells"])
    (OUT/"samples_verified.json").write_text(json.dumps(samples, indent=1, ensure_ascii=False))
else:
    p = OUT/"samples_verified.json"
    if p.exists(): samples = json.loads(p.read_text())
    print("amostras verificadas:", sum(1 for s in samples if s.get("compiles")))


## 6. Planilhas: rotulada (com metadados) e anonimizada (so o codigo)

Embaralhamos as 20 amostras e geramos dois CSVs: um **rotulado** (id original, rotulo,
gatilho, payload, degrau, resultado da verificacao) e um **anonimo** (id embaralhado + codigo
HDL), este ultimo e o que vai para o outro grupo detectar.

In [ ]:
import csv, random
random.seed(42)
def build_sheets(samples):
    order = samples[:]; random.shuffle(order)
    anon_map = {}
    # planilha rotulada
    with open(OUT/"planilha_rotulada.csv","w",newline="") as f:
        w = csv.writer(f)
        w.writerow(["anon_id","orig_id","label","trigger","payload","kind","step_name",
                    "compiles","tb_pass","cells","desc"])
        for i,s in enumerate(order, 1):
            aid = f"S{i:02d}"; anon_map[aid] = s["id"]
            w.writerow([aid, s["id"], s["label"], s.get("trigger",""), s.get("payload",""),
                        s.get("kind",""), s.get("step_name",""), s.get("compiles",""),
                        s.get("tb_pass",""), s.get("cells",""), s.get("desc","")])
    # planilha anonima (so codigo)
    with open(OUT/"planilha_anonima.csv","w",newline="") as f:
        w = csv.writer(f); w.writerow(["anon_id","hdl_code"])
        for i,s in enumerate(order, 1):
            w.writerow([f"S{i:02d}", s["code"]])
    (OUT/"anon_map.json").write_text(json.dumps(anon_map, indent=1))
    print("planilhas geradas:", len(order), "amostras")
    return anon_map
if RUN or not (OUT/"anon_map.json").exists():
    anon_map = build_sheets(samples)
else:
    anon_map = json.loads((OUT/"anon_map.json").read_text())


## 7. Deteccao com LLM (detector != gerador)

Enquanto o grupo de troca nao e definido, rodamos o detector sobre a **nossa propria** base
anonima como *self-check* (baseline do blue team). Quando chegar a planilha anonima do outro
grupo, basta apontar `DETECT_INPUT` para o CSV deles e rodar esta celula.

In [ ]:
import csv
DETECT_INPUT = OUT/"planilha_anonima.csv"  # trocar pelo CSV do outro grupo quando houver
def run_detect(llm, rows):
    res = []
    for r in rows:
        out = chat(llm, PR.SYS_DETECT, PR.prompt_detect(r["hdl_code"]),
                   max_tokens=1024, temperature=0.0)
        m = re.search(r"VERDICT:\s*(trojan|legit)", out, re.I)
        c = re.search(r"CONFIDENCE:\s*([0-9.]+)", out)
        res.append({"anon_id": r["anon_id"],
                    "pred": (m.group(1).lower() if m else "legit"),
                    "conf": float(c.group(1)) if c else 0.5,
                    "rationale": out.strip()[-500:]})
        print(r["anon_id"], "->", res[-1]["pred"], res[-1]["conf"])
    return res

if RUN:
    rows = list(csv.DictReader(open(DETECT_INPUT)))
    det = make_llm(DET_MODEL)
    detections = run_detect(det, rows)
    free(det)
    (OUT/"deteccao.json").write_text(json.dumps(detections, indent=1, ensure_ascii=False))
else:
    p = OUT/"deteccao.json"
    detections = json.loads(p.read_text()) if p.exists() else []
    print("deteccoes carregadas:", len(detections))


## 8. Metricas (acuracia, recall, FP, FN) e figuras

In [ ]:
# Casa predicoes com o gabarito via anon_map; calcula a matriz de confusao.
def metrics(detections, anon_map, samples):
    lab = {s["id"]: s["label"] for s in samples}
    TP=FP=FN=TN=0; rows=[]
    for d in detections:
        gold = lab.get(anon_map.get(d["anon_id"]))
        if gold is None: continue
        pred = d["pred"]
        TP += (gold=="trojan" and pred=="trojan")
        FN += (gold=="trojan" and pred=="legit")
        FP += (gold=="legit"  and pred=="trojan")
        TN += (gold=="legit"  and pred=="legit")
        rows.append((d["anon_id"], gold, pred, d["conf"]))
    n = TP+FP+FN+TN
    acc = (TP+TN)/n if n else 0
    rec = TP/(TP+FN) if (TP+FN) else 0
    prec= TP/(TP+FP) if (TP+FP) else 0
    m = {"TP":TP,"FP":FP,"FN":FN,"TN":TN,"n":n,"accuracy":acc,"recall":rec,"precision":prec}
    return m, rows

if detections:
    m, rows = metrics(detections, anon_map, samples)
    (OUT/"metricas.json").write_text(json.dumps(m, indent=1, ensure_ascii=False))
    print(json.dumps(m, indent=1))
else:
    print("sem deteccoes ainda — rode a celula 7 (RUN=True) ou carregue deteccao.json")


In [ ]:
# Figuras do relatorio (geradas a partir dos artefatos salvos).
if (OUT/"metricas.json").exists():
    os.system(f"cd {ROOT} && python outros/aula-08/figures.py {ROOT}")
    print("figuras em images/aula-08/")


## Notas

- **Config oficial do grupo:** gerador `Qwen2.5-Coder-14B-Instruct-AWQ`, detector
  `deepseek-coder-v2-lite-instruct-awq` (familia diferente). Todos devem usar a mesma config
  da vitima para comparabilidade entre as trocas de planilha.
- A deteccao sobre a propria base e um *self-check*; os numeros que vao ao relatorio sao os
  da troca com o outro grupo (apontar `DETECT_INPUT` para o CSV deles).
- A verificacao (iverilog/yosys) confirma que os trojans sao furtivos (passam no testbench
  quando dormentes); amostras que nao compilam sao regeneradas.
